# testing new memes

In [21]:
import requests
from IPython.display import display, Image, HTML
import os
from dotenv import load_dotenv
USERNAME = os.getenv('IMGFLIP_USERNAME')
PASSWORD = os.getenv('IMGFLIP_PASSWORD')
URL = "https://api.imgflip.com/caption_image"

In [22]:
def test_meme(input_dict):
    # append creds
    input_dict['username'] = USERNAME
    input_dict['password'] = PASSWORD
    

    # post and raise if issue
    response = requests.post(URL, data=input_dict)
    response.raise_for_status()

    # Parse the JSON response
    result = response.json()
    image_url = result['data']['url']

    # display in notebook
    display(Image(url=image_url))

In [23]:
test_meme(
{
    "template_id": "27813981",
    "text0": "When they open Powerpoint and you see slide '1 out of 243'",
    "text1": ""
  }




)

In [24]:
# from 10 to 27
import json
with open('../apps/gradio/memes20250304.json','r') as file:
    memes = json.load(file)

len(memes)

27

# we have more memes, so what? Let's try evaluating
- I will attempt to create a harness to see if memes improve scores
- We previously evaluated against 29 prompts using 4 binary criteria - relevant, surprising, creative, like_it - and one comment text entry
- Some researchers have attempted humour, creativity, shareability - https://arxiv.org/pdf/2501.11433
- Regardless excel is awful - so let's try jupyter notebook and maybe save to a local database

In [25]:
# get prompts
import pickle
with open('../data/evals/prompts.pkl', 'rb') as file:
    prompts = pickle.load(file)

len(prompts)

29

In [32]:
# 1. SETUP AND CONFIGURATION
import os
import json
import random
import pickle
import requests
import sqlite3
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import display, Image, HTML
import ipywidgets as widgets
import google.generativeai as genai

# Load environment variables
load_dotenv()

# load files
app_path = '../apps/gradio/'
with open(app_path+'system_prompt.txt', 'r') as f:
    system_prompt = f.read()

with open(app_path+'memes20250128.pkl', 'rb') as f:
    meme_context = pickle.load(f)


# Configure Gemini
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_prompt,
)

# Create or connect to SQLite database
def setup_database():
    conn = sqlite3.connect('../data/evals/meme_evaluations.db')
    cursor = conn.cursor()
    
    # Create table if it doesn't exist
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS meme_evaluations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        test_title TEXT NOT NULL,
        timestamp TEXT NOT NULL,
        prompt TEXT NOT NULL,
        gemini_json TEXT,
        imgflip_url TEXT,
        is_humorous INTEGER,
        is_creative INTEGER,
        is_shareable INTEGER,
        text_comments TEXT
    )
    ''')
    
    conn.commit()
    return conn, cursor

# Initialize database
conn, cursor = setup_database()
print("Database initialized successfully.")

# Helper function to clean JSON responses
def clean_response(response: str) -> str:
    """
    Cleans AI response to ensure valid JSON formatting.
    Handles edge cases with 'json' prefix and smart quote issues.
    """
    # Remove 'json' prefix and newlines at start
    cleaned = response.lstrip('json\n')
    
    # Remove markdown code blocks and backticks
    cleaned = cleaned.strip('`').replace('```json', '').replace('```', '')
    
    # Replace various quote types with standard double quotes
    cleaned = cleaned.replace('"', '"').replace('"', '"').replace("'", '"')
    
    # Handle escaped quotes inside text strings
    cleaned = cleaned.replace('\\"', '"')
    
    # Remove any trailing commas before closing braces
    cleaned = cleaned.replace(',}', '}')
    cleaned = cleaned.replace(',]', ']')
    
    return cleaned.strip()

Database initialized successfully.


In [27]:
# 2. PROMPT MANAGEMENT
# Function to load evaluation prompts
def load_evaluation_prompts(file_path=None):
    """
    Load evaluation prompts from a file or create a sample set.
    Returns a DataFrame with prompts.
    """
    if file_path and os.path.exists(file_path):
        try:
            with open(file_path, 'rb') as file:
                prompts = pickle.load(file)
            df = pd.DataFrame({"prompt":prompts})
            return df
        except Exception as e:
            print(f"Error loading prompts file: {e}")
    
    # Create a sample set if no file provided or loading failed
    sample_prompts = [
        "A meme about coding at 3am",
        "When your AI model finally works after days of debugging",
        "Data scientists trying to explain their job to family",
        "That moment when your code works on the first try",
        "Machine learning expectations vs reality"
    ]
    
    return pd.DataFrame({"prompt": sample_prompts})

# Function to initialize a new test session
def initialize_test_session(test_title):
    """
    Initialize a new test session with the given title.
    Returns a DataFrame for tracking progress.
    """
    prompts_df = load_evaluation_prompts()
    
    # Add columns for tracking progress and results
    prompts_df['test_title'] = test_title
    prompts_df['evaluated'] = False
    prompts_df['gemini_json'] = None
    prompts_df['imgflip_url'] = None
    
    return prompts_df

# Sample usage
# test_session = initialize_test_session("Test Session 1")
# display(test_session.head())

# Create a test session manager
class TestSessionManager:
    def __init__(self, test_title):
        self.test_title = test_title
        self.session_df = initialize_test_session(test_title)
        self.current_index = 0
        print(f"Test session '{test_title}' initialized with {len(self.session_df)} prompts.")
    
    def get_current_prompt(self):
        """Get the current prompt to evaluate."""
        if self.current_index < len(self.session_df):
            return self.session_df.iloc[self.current_index]["prompt"]
        return None
    
    def move_to_next_prompt(self):
        """Move to the next prompt."""
        self.current_index += 1
        if self.current_index < len(self.session_df):
            return self.session_df.iloc[self.current_index]["prompt"]
        else:
            print("All prompts have been processed.")
            return None
    
    def update_current_result(self, gemini_json, imgflip_url):
        """Update results for the current prompt."""
        if self.current_index < len(self.session_df):
            self.session_df.at[self.current_index, 'gemini_json'] = json.dumps(gemini_json)
            self.session_df.at[self.current_index, 'imgflip_url'] = imgflip_url
    
    def mark_current_evaluated(self):
        """Mark the current prompt as evaluated."""
        if self.current_index < len(self.session_df):
            self.session_df.at[self.current_index, 'evaluated'] = True
    
    def get_progress(self):
        """Get progress statistics."""
        total = len(self.session_df)
        evaluated = sum(self.session_df['evaluated'])
        return {
            'current_index': self.current_index,
            'total_prompts': total,
            'evaluated_prompts': evaluated,
            'progress_percentage': round(evaluated / total * 100 if total > 0 else 0, 2)
        }

# Initialize test session
# session_manager = TestSessionManager("Test Session 1")

# todo - evaluate the following

In [29]:
# 3. MEME GENERATION PIPELINE

def construct_meme_prompt(user_input, meme_context=None, previous_attempts=None):
    """
    Construct a prompt for the meme generation model.
    Simplified version of the function in the provided code.
    """
    
    # Shuffle the meme context
    shuffled_context = list(meme_context)
    random.shuffle(shuffled_context)
    
    sections = []
    
    # Add failed attempts if they exist
    if previous_attempts:
        sections.append("PREVIOUS ATTEMPTS (TRY A DIFFERENT TEMPLATE):")
        sections.extend(str(attempt) for attempt in previous_attempts)
    
    sections.extend([
        f"USER INPUT: {user_input.strip()}",
        f"AVAILABLE CONTEXT: {str(shuffled_context)}"
    ])

    return "\n\n".join(sections)

def generate_meme_completion(prompt, model, max_attempts=3):
    """
    Generate a meme completion using the Gemini model.
    Returns a JSON object with meme information.
    """
    config = {
        'max_output_tokens': 1000,
        'temperature': 0.1,
    }
    
    last_error = None
    for attempt in range(max_attempts):
        try:
            # Generate completion
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(**config)
            )
            
            if not response.text:
                last_error = RuntimeError("Model returned empty response")
                continue
            
            # Clean the response text
            cleaned_response = clean_response(response.text)
            
            # Parse JSON response
            meme_data = json.loads(cleaned_response)
            
            # Ensure required keys exist
            required_keys = ['template_id', 'text0', 'text1']
            missing_keys = [key for key in required_keys if key not in meme_data]
            
            if missing_keys:
                raise ValueError(f"Missing required keys: {missing_keys}")
            
            return meme_data
            
        except (json.JSONDecodeError, ValueError) as e:
            last_error = RuntimeError(f"Invalid JSON in model response (attempt {attempt + 1}/{max_attempts}): {e}")
            continue
            
    # If we get here, all attempts failed
    raise RuntimeError(f"Failed after {max_attempts} attempts. Last error: {last_error}")

def create_imgflip_meme(meme_data):
    """
    Create a meme using the imgflip API.
    Returns the result with the meme URL.
    """
    api_url = "https://api.imgflip.com/caption_image"
    
    # Add credentials
    data = meme_data.copy()
    data.update({
        'username': os.getenv('IMGFLIP_USERNAME'),
        'password': os.getenv('IMGFLIP_PASSWORD')
    })
    
    try:
        response = requests.post(api_url, data=data)
        response_data = response.json()
        
        if response.status_code == 200 and response_data.get('success'):
            return {
                'success': True,
                'image_url': response_data['data']['url'],
                'raw_response': response_data
            }
        else:
            error_msg = response_data.get('error_message', 'Unknown error')
            return {
                'success': False,
                'error_message': error_msg,
                'raw_response': response_data
            }
            
    except Exception as e:
        raise RuntimeError(f"API request failed: {str(e)}")

def process_meme_request(prompt_text, meme_context=None):
    """
    Process a meme request from prompt to display.
    Returns the Gemini JSON and ImgFlip response.
    """
    # Step 1: Construct the prompt
    meme_prompt = construct_meme_prompt(prompt_text, meme_context)
    
    try:
        # Step 2: Generate meme JSON
        meme_json = generate_meme_completion(meme_prompt, model)
        
        # Step 3: Create meme via ImgFlip
        imgflip_response = create_imgflip_meme(meme_json)
        
        if imgflip_response['success']:
            # Display the meme
            display(Image(url=imgflip_response['image_url']))
            return meme_json, imgflip_response
        else:
            print(f"Error creating meme: {imgflip_response.get('error_message', 'Unknown error')}")
            return meme_json, imgflip_response
    
    except Exception as e:
        print(f"Error processing meme request: {str(e)}")
        return None, None

# Example usage:
# meme_json, imgflip_response = process_meme_request("When your code works on the first try")

In [30]:
# 4. EVALUATION INTERFACE
def create_evaluation_widgets():
    """
    Create widgets for the evaluation interface.
    Returns the widgets and a function to get their values.
    """
    # Title
    title = widgets.HTML(value="<h2>Meme Evaluation</h2>")
    
    # Current prompt display
    prompt_display = widgets.HTML(value="<b>Current Prompt:</b> ")
    
    # Rating widgets
    humor_label = widgets.HTML(value="<b>Humor Rating (1-5):</b>")
    humor_slider = widgets.IntSlider(min=1, max=5, step=1, value=3, description='')
    
    creativity_label = widgets.HTML(value="<b>Creativity Rating (1-5):</b>")
    creativity_slider = widgets.IntSlider(min=1, max=5, step=1, value=3, description='')
    
    shareability_label = widgets.HTML(value="<b>Shareability Rating (1-5):</b>")
    shareability_slider = widgets.IntSlider(min=1, max=5, step=1, value=3, description='')
    
    # Comments widget
    comments_label = widgets.HTML(value="<b>Comments:</b>")
    comments_text = widgets.Textarea(
        value='',
        placeholder='Enter your comments about this meme...',
        description='',
        layout=widgets.Layout(width='100%', height='100px')
    )
    
    # Submit button
    submit_button = widgets.Button(
        description='Submit Evaluation',
        button_style='success',
        icon='check'
    )
    
    # Progress display
    progress_display = widgets.HTML(value="")
    
    # Group widgets
    widgets_dict = {
        'title': title,
        'prompt_display': prompt_display,
        'humor_label': humor_label,
        'humor_slider': humor_slider,
        'creativity_label': creativity_label,
        'creativity_slider': creativity_slider,
        'shareability_label': shareability_label,
        'shareability_slider': shareability_slider,
        'comments_label': comments_label,
        'comments_text': comments_text,
        'submit_button': submit_button,
        'progress_display': progress_display
    }
    
    # Function to get evaluation values
    def get_evaluation_values():
        return {
            'is_humorous': humor_slider.value,
            'is_creative': creativity_slider.value,
            'is_shareable': shareability_slider.value,
            'text_comments': comments_text.value
        }
    
    return widgets_dict, get_evaluation_values

def save_evaluation_to_db(conn, cursor, test_title, prompt, gemini_json, imgflip_url, evaluation_data):
    """
    Save the evaluation data to the SQLite database.
    """
    try:
        cursor.execute('''
        INSERT INTO meme_evaluations 
        (test_title, timestamp, prompt, gemini_json, imgflip_url, 
         is_humorous, is_creative, is_shareable, text_comments)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            test_title,
            datetime.now().isoformat(),
            prompt,
            json.dumps(gemini_json) if gemini_json else None,
            imgflip_url,
            evaluation_data['is_humorous'],
            evaluation_data['is_creative'],
            evaluation_data['is_shareable'],
            evaluation_data['text_comments']
        ))
        
        conn.commit()
        return True
    except Exception as e:
        print(f"Error saving to database: {str(e)}")
        return False

def reset_evaluation_form(widgets_dict):
    """
    Reset the evaluation form to default values.
    """
    widgets_dict['humor_slider'].value = 3
    widgets_dict['creativity_slider'].value = 3
    widgets_dict['shareability_slider'].value = 3
    widgets_dict['comments_text'].value = ''

def update_progress_display(widgets_dict, progress_data):
    """
    Update the progress display with current statistics.
    """
    progress_html = f"""
    <div style="margin-top: 10px; padding: 10px; background-color: #f0f0f0; border-radius: 5px;">
        <b>Progress:</b> {progress_data['evaluated_prompts']}/{progress_data['total_prompts']} 
        ({progress_data['progress_percentage']}% complete)
    </div>
    """
    widgets_dict['progress_display'].value = progress_html

In [31]:
# 5. MAIN EVALUATION LOOP
def initialize_evaluation_session(test_title):
    """
    Initialize a complete evaluation session.
    Returns all necessary components for the evaluation.
    """
    # Initialize database
    conn, cursor = setup_database()
    
    # Initialize session manager
    session_manager = TestSessionManager(test_title)
    
    # Create evaluation widgets
    widgets_dict, get_evaluation_values = create_evaluation_widgets()
    
    # Initialize meme data storage
    current_meme = {
        'gemini_json': None,
        'imgflip_url': None
    }
    
    # Update progress display
    update_progress_display(widgets_dict, session_manager.get_progress())
    
    # Function to process the next prompt
    def process_next_prompt():
        prompt = session_manager.get_current_prompt()
        if prompt:
            widgets_dict['prompt_display'].value = f"<b>Current Prompt:</b> {prompt}"
            
            # Process the meme request
            meme_json, imgflip_response = process_meme_request(prompt)
            
            if meme_json and imgflip_response and imgflip_response.get('success'):
                # Store the current meme data
                current_meme['gemini_json'] = meme_json
                current_meme['imgflip_url'] = imgflip_response['image_url']
                
                # Update session manager
                session_manager.update_current_result(meme_json, imgflip_response['image_url'])
                
                return True
            else:
                print("Failed to generate meme. Skipping to next prompt.")
                session_manager.move_to_next_prompt()
                return process_next_prompt()
        else:
            widgets_dict['prompt_display'].value = "<b>All prompts evaluated!</b>"
            return False
    
    # Function to handle submit button click
    def on_submit_button_clicked(b):
        # Get evaluation values
        evaluation_data = get_evaluation_values()
        
        # Save to database
        prompt = session_manager.get_current_prompt()
        save_evaluation_to_db(
            conn, cursor, 
            test_title, 
            prompt, 
            current_meme['gemini_json'], 
            current_meme['imgflip_url'], 
            evaluation_data
        )
        
        # Mark as evaluated
        session_manager.mark_current_evaluated()
        
        # Move to next prompt
        session_manager.move_to_next_prompt()
        
        # Reset the form
        reset_evaluation_form(widgets_dict)
        
        # Update progress
        update_progress_display(widgets_dict, session_manager.get_progress())
        
        # Process the next prompt
        process_next_prompt()
    
    # Attach the click handler
    widgets_dict['submit_button'].on_click(on_submit_button_clicked)
    
    # Process the first prompt
    process_next_prompt()
    
    # Display the evaluation interface
    display(widgets_dict['title'])
    display(widgets_dict['prompt_display'])
    display(widgets_dict['humor_label'])
    display(widgets_dict['humor_slider'])
    display(widgets_dict['creativity_label'])
    display(widgets_dict['creativity_slider'])
    display(widgets_dict['shareability_label'])
    display(widgets_dict['shareability_slider'])
    display(widgets_dict['comments_label'])
    display(widgets_dict['comments_text'])
    display(widgets_dict['submit_button'])
    display(widgets_dict['progress_display'])
    
    return conn, cursor, session_manager

# Example usage:
# conn, cursor, session_manager = initialize_evaluation_session("Test Session 1")

In [33]:
# 7. COMPLETE USAGE EXAMPLE
# Copy and run this cell to start a new evaluation session

def start_new_evaluation_session():
    """
    Start a new evaluation session with a user-provided test title.
    """
    # Get test title
    test_title = input("Enter a title for this test session: ")
    
    print(f"Starting evaluation session: {test_title}")
    print("Loading resources and initializing...")
    
    # Initialize evaluation session
    conn, cursor, session_manager = initialize_evaluation_session(test_title)
    
    print("Evaluation session started. Complete the evaluations using the interface above.")
    print("After completing all evaluations, run the analysis cells to view results.")
    
    return conn, cursor, session_manager



In [ ]:
# Uncomment to start a new session:
conn, cursor, session_manager = start_new_evaluation_session()